# MDAnalysis Bridge — RMSF Parity with analysis.js (I83)

Load exported DCD/PDB in MDAnalysis and compute RMSF that matches `src/analysis.js`.
Formula: `B = (8π²/3) * <Δr²>` where `<Δr²> = RMSF²` (`src/analysis.js:371`).


In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis.rms import RMSF
from MDAnalysis.analysis import align
import numpy as np
from scipy.stats import pearsonr

# Export from browser: Recording panel → Format=PDB → Download
u = mda.Universe("cg_traj_500frames.pdb")  # MODEL/ENDMDL PDB
ca = u.select_atoms("name CA")
print(f"{len(ca)} Cα, {len(u.trajectory)} frames")

# Kabsch align to first frame (matches src/analysis.js:kabsch)
align.AlignTraj(u, u, select="name CA", in_memory=True).run()

# RMSF per residue (Å)
rmsf = RMSF(ca).run()
rmsf_vals = rmsf.results.rmsf

# B-factors matching src/analysis.js:371 RMSF_TO_B = 8π²/3 ≈ 26.32
B_sim_mda = (8 * np.pi**2 / 3.0) * (rmsf_vals ** 2)
print(B_sim_mda[:5])

B_exp = ca.tempfactors
mask = B_exp > 0
if mask.sum() >= 2:
    r, p = pearsonr(B_sim_mda[mask], B_exp[mask])
    print(f"Pearson r(B_sim, B_exp) = {r:.3f} over {mask.sum()} residues")

## Notes
- Always align first — otherwise RMSF is inflated by diffusion.
- `B_sim (analysis.js) ≈ (8π²/3) * RMSF(MDA)²` within Float32 round-off.
- See `docs/BRIDGE.md` for full DCD/XYZ options and troubleshooting.